In [ ]:
# hide
# no-output
from IPython.utils.capture import capture_output
with capture_output():
    %pip install -q plotly anywidget

import asyncio
import os
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import Audio
import icm_plotly
from icm_plotly import RED, BLUE, GOLD, IRON, TEAL, STEEL

Drag the delay $M$ and the gain $g$ of $y[n] = x[n] + g \cdot y[n - M]$. On
the left is the first 25 ms of its impulse response, an impulse train that
decays by $g$ every $M$ samples. On the right is its magnitude response, a
comb whose teeth sit at the multiples of $\frac{f_s}{M}$ (dotted). The audio
card plays two seconds of the impulse response.

In [ ]:
# hide
# autorun
M0, G0 = 100, 0.8                   # the chapter's example

SR = 44100
T_IR = 0.025                        # show the first 25 ms of the impulse response
F_MAX = 4500.0
N_PLAY = int(2.0 * SR)              # the audio card plays two seconds


def stems(n, v):
    """x and y arrays that draw stems from 0 to v at each n, as one trace."""
    xs = np.repeat(np.asarray(n, float), 3)
    ys = np.column_stack([np.zeros(len(v)), v, np.full(len(v), np.nan)]).ravel()
    xs[2::3] = np.nan
    return xs, ys


def impulse_response(M, g, SR=SR, T_IR=T_IR):
    k = np.arange(int(T_IR * SR) // M + 1)
    return k * M / SR * 1000, g ** k    # times in ms, heights g^k


def response_db(M, g, SR=SR, F_MAX=F_MAX):
    # a fine grid, plus every tooth exactly, so no peak falls between points
    teeth = np.arange(0.0, F_MAX, SR / M)
    f = np.union1d(np.linspace(0.0, F_MAX, 3000), teeth)
    H = 1 / (1 - g * np.exp(-2j * np.pi * f * M / SR))
    return f, 20 * np.log10(np.abs(H))


def tooth_lines(M, SR=SR, F_MAX=F_MAX):
    xs = []
    for f in np.arange(SR / M, F_MAX, SR / M):
        xs += [f, f, None]
    return xs, [-12, 42, None] * (len(xs) // 3)


def figure():
    fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.12)
    t, h = impulse_response(M0, G0)
    xs, ys = stems(t, h)
    fig.add_scatter(x=xs, y=ys, mode="lines", line=dict(color=RED, width=1.6),
                    row=1, col=1)
    fig.add_scatter(x=t, y=h, mode="markers", marker=dict(color=RED, size=6),
                    row=1, col=1)
    lx, ly = tooth_lines(M0)
    fig.add_scatter(x=lx, y=ly, mode="lines",
                    line=dict(color=STEEL, width=1, dash="dot"), row=1, col=2)
    f, db = response_db(M0, G0)
    fig.add_scatter(x=f, y=db, mode="lines", line=dict(color=RED, width=1.8),
                    row=1, col=2)
    fig.update_xaxes(range=[-0.4, T_IR * 1000], title_text="Time (ms)",
                     fixedrange=True, row=1, col=1)
    fig.update_yaxes(range=[0, 1.08], title_text="h[n]", fixedrange=True,
                     row=1, col=1)
    fig.update_xaxes(range=[0, F_MAX], title_text="Frequency (Hz)",
                     fixedrange=True, row=1, col=2)
    fig.update_yaxes(range=[-12, 42], title_text="Magnitude (dB)", fixedrange=True,
                     row=1, col=2)
    return fig


def controls(fig):
    M = widgets.IntSlider(description="Delay $M$ (samples)", min=10, max=400,
                          value=M0)
    g = widgets.FloatSlider(description="Gain $g$", min=0.0, max=0.99,
                            step=0.01, value=G0)
    readout = widgets.HTML()

    # the defaults snapshot the arrays; the page's notebooks share one kernel
    def update(M, g, stems=stems, impulse_response=impulse_response,
               response_db=response_db, tooth_lines=tooth_lines, SR=SR,
               readout=readout):
        t, h = impulse_response(M, g)
        xs, ys = stems(t, h)
        f, db = response_db(M, g)
        lx, ly = tooth_lines(M)
        with fig.batch_update():
            fig.data[0].x, fig.data[0].y = xs, ys
            fig.data[1].x, fig.data[1].y = t, h
            fig.data[2].x, fig.data[2].y = lx, ly
            fig.data[3].x, fig.data[3].y = f, db
        if g > 0:
            T60 = 3 * M / (SR * -np.log10(g))
            decay = f"decays by 60 dB in <i>T</i> = {T60:.2f} s"
        else:
            decay = "no feedback, so no echoes at all"
        readout.value = (
            f"<span style='font-size:0.9em'><i>f</i><sub>s</sub> / <i>M</i> = "
            f"{SR / M:.1f} Hz &nbsp;·&nbsp; teeth reach "
            f"+{-20 * np.log10(1 - g):.1f} dB, valleys "
            f"&minus;{20 * np.log10(1 + g):.1f} dB &nbsp;·&nbsp; {decay}</span>")

    widgets.interactive_output(update, {"M": M, "g": g})

    # the audio card under the controls: the previous clip stays in place
    # while you drag (so the layout never jumps) and is swapped for the new
    # one when the pointer releases (keyboard nudges settle on a timer). It is
    # written through the Output's synced `outputs` trait, which works
    # outside a kernel message, where display() output has no destination
    out = widgets.Output()
    gate = icm_plotly.release_gate()   # pointer state: is a slider mid-drag?
    pending = []
    dirty = []

    def show_audio(x, SR=SR):
        audio = Audio(x.astype(np.float32), rate=SR, normalize=False)
        data, metadata = get_ipython().display_formatter.format(audio)
        # one assignment swaps the old card for the new one in place, so
        # the page never shows an empty card and nothing shifts
        out.outputs = ({"output_type": "display_data",
                        "data": data, "metadata": metadata},)

    def render(N_PLAY=N_PLAY):
        y = np.zeros(N_PLAY)
        y[::M.value] = g.value ** np.arange(len(y[::M.value]))
        show_audio(0.5 * y)

    async def settle():
        await asyncio.sleep(0.25)
        pending.clear()
        if dirty and not gate.dragging:
            dirty.clear()
            render()

    def on_change(_):
        dirty.append(True)
        if pending:
            pending.pop().cancel()
        pending.append(asyncio.ensure_future(settle()))

    def on_release(change):
        if not change["new"] and dirty:
            if pending:
                pending.pop().cancel()
            dirty.clear()
            render()

    gate.observe(on_release, names="dragging")

    for s in (M, g):
        s.observe(on_change, names="value")
    if not os.environ.get("ICM_BOOK_BUILD"):   # the build bakes no card
        render()
    return widgets.VBox([M, g, readout, out, gate])

icm_plotly.show(figure, controls)